# Direction-Agnostic EURUSD H1 Model

**Purpose:** Train a single model that predicts **which direction** EURUSD will break
over the next 6 H1 hours — UP or DOWN — using a symmetrical risk envelope. The model
replaces the separate BUY and SELL models with one architecture that can trade in
**either** regime.

> **Why this exists:** the existing SELL-only pipeline goes silent during USD-weak
> regimes. A direction-agnostic model reads the same features and answers: *"given the
> current conditions, will price hit an upper target before a lower target?"* The
> system enters in whatever direction the model predicts — BUY when UP, SELL when DOWN.

**Target (`Y`):** binary — `1` if price hits the **upper** ATR-scaled target before
the **lower** target, `0` otherwise. Both targets are the same distance from entry
(`R × ATR`), making the base rate naturally balanced near 50/50.

**Pipeline overview:**
1. Configuration (paths, R envelope, split dates)
2. Load + validate raw H1 OHLCV data
3. Feature engineering (90+ H1 indicators, same as SELL model)
4. Label generation (symmetrical UP/DOWN, sweep R values)
5. Chronological split → train / val / test (test sealed)
6. Feature selection (noise-injection voting, same method)
7. Model selection (purged nested CV + recency weights, optimizing **ROC-AUC**)
8. Final model (isotonic calibration + soft-vote ensemble)
9. Sealed test evaluation (ROC-AUC / accuracy / calibration / direction-margin)
10. Save bundle to `models_bin/`


In [ ]:
# ── Core imports ────────────────────────────────────────────────────────────────
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
%matplotlib inline

print(f"Python  : {sys.version}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")


## 1. Configuration

All paths and key parameters live in one place. Change `R_ENVELOPE`, the split dates, or
the forward horizon here — every downstream cell reads from these.

### The R-envelope parameter

`R_ENVELOPE` is the symmetrical risk distance in ATR units. Both the upper target and
lower target are the same distance from entry:

```
tp_up   = entry + ATR × R_ENVELOPE
tp_down = entry - ATR × R_ENVELOPE
```

- `R = 1.0` → tight envelope, more signals, harder to predict (small moves)
- `R = 1.5` → wide envelope, fewer signals, easier to predict (bigger moves)

Section 5 sweeps over R values to pick the best balance. Start with `R = 1.0` and adjust
upward only if the model fails to find signal.


In [ ]:
from pathlib import Path

# ── Instrument & timeframe ────────────────────────────────────────────────────
PAIR      = "AGNOSTIC"  # placeholder — trained on EURUSD H1 data
TIMEFRAME = "H1"

# ── Data paths ────────────────────────────────────────────────────────────────
ROOT = Path("../..").resolve()

RAW_DATA_DIR  = ROOT / "data" / "raw" / "mt5" / TIMEFRAME
RAW_FILE      = RAW_DATA_DIR / "EURUSD_H1.csv"  # trained on EURUSD data

MODELS_DIR    = ROOT / "models_bin"
FEATURES_DIR  = ROOT / "data" / "features"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Label parameters ──────────────────────────────────────────────────────────
# Symmetrical envelope: both targets are R_ENVELOPE × ATR from entry.
# Swept over R values in Section 5 to find the best trainable balance.
ATR_PERIOD   = 14      # bars used to compute ATR(14)
R_ENVELOPE   = 1.0     # symmetrical risk distance (ATR units)
FORWARD_BARS = 6       # how many H1 bars forward to scan

# ── Chronological split boundaries ───────────────────────────────────────────
TRAIN_START = "2020-06-30 00:00:00"
TRAIN_END   = "2025-06-30 23:00:00"
VAL_START   = "2025-07-01 00:00:00"
VAL_END     = "2025-12-31 23:00:00"
TEST_START  = "2026-01-01 00:00:00"

# ── CV / training knobs ──────────────────────────────────────────────────────
OUTER_FOLDS   = 5
INNER_FOLDS   = 4
PURGE_DAYS    = 30
SEARCH_ITERS  = 25     # reduced for memory safety
RECENCY_DECAY = 0.15

print(f"Pair          : {PAIR} (trained on EURUSD data)")
print(f"Timeframe     : {TIMEFRAME}")
print(f"Raw file      : {RAW_FILE}  (exists={RAW_FILE.exists()})")
print(f"R envelope    : {R_ENVELOPE}")
print(f"Forward bars  : {FORWARD_BARS}")
print(f"Train         : {TRAIN_START}  →  {TRAIN_END}")
print(f"Val           : {VAL_START}  →  {VAL_END}")
print(f"Test          : {TEST_START}  →  present")


## 2. Load Raw Data

Standard EURUSD H1 OHLCV bars from MT5. Same data source as the SELL model.


In [ ]:
df = pd.read_csv(RAW_FILE, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

print(f"Rows      : {len(df):,}")
print(f"Columns   : {list(df.columns)}")
print(f"Date range: {df['datetime'].min()}  →  {df['datetime'].max()}")
print(f"Nulls     : {df.isnull().sum().sum()}")


## 3. Target variable — `y_agnostic` (the UP/DOWN label)

### What we predict

For every H1 bar we ask: *"over the next FORWARD_BARS hours, will price hit the UPPER
target before the LOWER target?"*

Both targets are exactly `R_ENVELOPE × ATR` away from the entry price:

$$\text{entry} = \text{close}[t]$$
$$\text{tp\_up} = \text{close}[t] + \text{ATR}[t] \times R\_ENVELOPE$$
$$\text{tp\_down} = \text{close}[t] - \text{ATR}[t] \times R\_ENVELOPE$$

The label is **1** if the upper target is reached before the lower target, **0** otherwise.
If neither is reached within `FORWARD_BARS` bars, the label is 0 (no edge).

### Why symmetrical, not directional

The existing SELL model asks: *"will price fall 1.5× ATR before rising 1.0× ATR?"*
That's an asymmetric question — the TP is farther than the SL, biasing the model toward
SELL in trending markets. This model asks a symmetric question — the targets are
equidistant — so the base rate is naturally balanced near 50/50.

A 50/50 base rate means:

- The model cannot "hide" behind a majority class. A coin-flip scores 0.5 ROC-AUC.
- If the model finds signal (ROC-AUC > 0.55), it's real directional edge, not
  regime-exploitation.
- The system enters in **whichever direction the model predicts** — BUY when UP,
  SELL when DOWN. No more regime lock.

### Class balance

Expected base rate near 50% — roughly half of all bars tip upward first, half downward.
The sweep in Section 5 confirms this across R values.


## 4. Feature Engineering

Identical to the EURUSD SELL model — 90+ H1 technical indicators, D1 context, calendar
merge, auxiliary macro symbols. The feature set does not change. Only the target changes.


In [ ]:
def compute_features(data: pd.DataFrame, pair: str = None) -> pd.DataFrame:
    """
    Compute all technical features from raw H1 OHLCV data.
    (Same function as the SELL notebook — included here for self-contained execution.)
    """
    df = data.copy()

    # ── 1. Time features ──────────────────────────────────────────────────────
    df["hour"]        = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek
    df["month"]       = df["datetime"].dt.month
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["day_of_week"] / 5)
    df["dow_cos"]  = np.cos(2 * np.pi * df["day_of_week"] / 5)
    _h = df["hour"]
    df["session_asian"]   = ((_h >= 0)  & (_h < 7)).astype(int)
    df["session_london"]  = ((_h >= 7)  & (_h < 16)).astype(int)
    df["session_ny"]      = ((_h >= 13) & (_h < 22)).astype(int)
    df["session_overlap"] = ((_h >= 13) & (_h < 17)).astype(int)

    # ── 2. Candle structure ───────────────────────────────────────────────────
    df["body_size"]   = abs(df["close"] - df["open"])
    df["price_range"] = df["high"] - df["low"]
    df["upper_wick"]  = df["high"] - df[["open", "close"]].max(axis=1)
    df["lower_wick"]  = df[["open", "close"]].min(axis=1) - df["low"]
    df["body_ratio"]  = df["body_size"] / df["price_range"].replace(0, np.nan)

    # ── 3. Moving averages ────────────────────────────────────────────────────
    for w in [10, 20, 50]:
        df[f"sma_{w}"] = df["close"].rolling(w).mean()
    for s in [10, 20, 50, 100, 200]:
        df[f"ema_{s}"] = df["close"].ewm(span=s, adjust=False).mean()
    df["close_vs_ema50"]  = (df["close"] - df["ema_50"])  / df["close"]
    df["close_vs_ema200"] = (df["close"] - df["ema_200"]) / df["close"]

    # ── 4. Momentum oscillators ───────────────────────────────────────────────
    delta   = df["close"].diff()
    gain    = delta.clip(lower=0)
    loss    = (-delta).clip(lower=0)
    avg_g   = gain.rolling(14).mean()
    avg_l   = loss.rolling(14).mean()
    rs      = avg_g / avg_l.replace(0, np.nan)
    df["rsi_14"] = 100 - (100 / (1 + rs))
    low14   = df["low"].rolling(14).min()
    high14  = df["high"].rolling(14).max()
    df["stoch_k"]    = 100 * (df["close"] - low14) / (high14 - low14).replace(0, np.nan)
    df["stoch_d"]    = df["stoch_k"].rolling(3).mean()
    df["williams_r"] = -100 * (high14 - df["close"]) / (high14 - low14).replace(0, np.nan)
    ema12              = df["close"].ewm(span=12, adjust=False).mean()
    ema26              = df["close"].ewm(span=26, adjust=False).mean()
    df["macd"]         = ema12 - ema26
    df["macd_sig"]     = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"]    = df["macd"] - df["macd_sig"]
    df["macd_hist_slope"] = df["macd_hist"].diff()

    # ── 5. Volatility ─────────────────────────────────────────────────────────
    high_low   = df["high"] - df["low"]
    high_pc    = (df["high"] - df["close"].shift()).abs()
    low_pc     = (df["low"]  - df["close"].shift()).abs()
    true_range = pd.concat([high_low, high_pc, low_pc], axis=1).max(axis=1)
    df["atr_14"] = true_range.rolling(ATR_PERIOD).mean()
    df["atr_regime"] = df["atr_14"] / df["atr_14"].rolling(50).mean()
    bb_mid          = df["close"].rolling(20).mean()
    bb_std          = df["close"].rolling(20).std()
    df["bb_upper"]  = bb_mid + 2 * bb_std
    df["bb_lower"]  = bb_mid - 2 * bb_std
    df["bb_width"]  = (df["bb_upper"] - df["bb_lower"]) / bb_mid
    df["bb_pct"]    = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"]).replace(0, np.nan)
    for w in [10, 20, 50]:
        df[f"rolling_std_{w}"] = df["close"].rolling(w).std()

    # ── 6. Trend strength ─────────────────────────────────────────────────────
    plus_dm  = df["high"].diff().clip(lower=0)
    minus_dm = (-df["low"].diff()).clip(lower=0)
    plus_dm  = plus_dm.where(plus_dm > minus_dm, 0)
    minus_dm = minus_dm.where(minus_dm > plus_dm, 0)
    atr_adx  = true_range.rolling(14).mean()
    plus_di  = 100 * (plus_dm.rolling(14).mean()  / atr_adx.replace(0, np.nan))
    minus_di = 100 * (minus_dm.rolling(14).mean() / atr_adx.replace(0, np.nan))
    dx       = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    df["adx_14"]   = dx.rolling(14).mean()
    df["plus_di"]  = plus_di
    df["minus_di"] = minus_di
    tp_ = (df["high"] + df["low"] + df["close"]) / 3
    df["cci_20"]     = (tp_ - tp_.rolling(20).mean()) / (0.015 * tp_.rolling(20).std())
    df["roc_10"]     = df["close"].pct_change(10) * 100
    df["momentum_10"] = df["close"] - df["close"].shift(10)

    # ── 7. Volume ─────────────────────────────────────────────────────────────
    df["obv"]          = (np.sign(df["close"].diff()) * df["volume"]).fillna(0).cumsum()
    vol_ma20           = df["volume"].rolling(20).mean()
    df["volume_ratio"] = df["volume"] / vol_ma20.replace(0, np.nan)
    obv_vals = df["obv"].values
    df["obv_slope_20"] = np.nan
    for i in range(20, len(obv_vals)):
        slope, _ = np.polyfit(range(20), obv_vals[i-20:i], 1)
        df.iloc[i, df.columns.get_loc("obv_slope_20")] = slope
    df["obv_zscore_100"] = (df["obv"] - df["obv"].rolling(100).mean()) / df["obv"].rolling(100).std()

    # ── 8. Lagged features ────────────────────────────────────────────────────
    for lag in [1, 2, 3, 5]:
        df[f"close_lag_{lag}"]  = df["close"].shift(lag)
        df[f"rsi_lag_{lag}"]    = df["rsi_14"].shift(lag)
        df[f"atr_lag_{lag}"]    = df["atr_14"].shift(lag)
        df[f"volume_lag_{lag}"] = df["volume"].shift(lag)

    # ── 9. Price-change returns ───────────────────────────────────────────────
    for p in [1, 5, 10]:
        df[f"return_{p}b"] = df["close"].pct_change(p)

    # ── 10. D1 context (resampled from H1, no leakage) ────────────────────────
    _d1 = (
        df.set_index("datetime")[["open", "high", "low", "close", "volume"]]
        .resample("1D").agg({"open": "first", "high": "max", "low": "min",
                              "close": "last", "volume": "sum"}).dropna()
    )
    _d1["d1_ema20"]  = _d1["close"].ewm(span=20, adjust=False).mean()
    _d1["d1_ema50"]  = _d1["close"].ewm(span=50, adjust=False).mean()
    _d1["d1_trend"]  = (_d1["d1_ema20"] > _d1["d1_ema50"]).astype(int)
    _dd = _d1["close"].diff()
    _d1["d1_rsi"]    = 100 - 100 / (
        1 + _dd.clip(lower=0).rolling(14).mean()
          / (-_dd).clip(lower=0).rolling(14).mean().replace(0, np.nan)
    )
    _d1["d1_close_vs_ema20"] = (_d1["close"] - _d1["d1_ema20"]) / _d1["close"]
    _d1_cols    = ["d1_ema20", "d1_ema50", "d1_trend", "d1_rsi", "d1_close_vs_ema20"]
    _d1_shifted = _d1[_d1_cols].shift(1).reset_index().rename(columns={"datetime": "_d1_date"})

    # ── 11. Intraday context ──────────────────────────────────────────────────
    df["_date"]   = df["datetime"].dt.normalize()
    _day_open     = df.groupby("_date")["open"].first().rename("_day_open")
    df            = df.join(_day_open, on="_date")
    df["close_vs_day_open"] = (df["close"] - df["_day_open"]) / df["_day_open"]
    df = df.merge(_d1_shifted, left_on="_date", right_on="_d1_date", how="left")
    df.drop(columns=["_date", "_d1_date", "_day_open"], inplace=True)

    # ── Drop NaN rows ─────────────────────────────────────────────────────────
    n_before = len(df)
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Rows dropped (NaN warm-up): {n_before - len(df):,}  |  Rows remaining: {len(df):,}")

    # ── Calendar features ─────────────────────────────────────────────────────
    if pair:
        try:
            import sys as _sys
            _root = Path('../../..').resolve()
            if str(_root / 'frival') not in _sys.path:
                _sys.path.insert(0, str(_root / 'frival'))
            from data.calendar import compute_calendar_features
            cal_feats = compute_calendar_features(df["datetime"], pair)
            for col in cal_feats.columns:
                df[col] = cal_feats[col].values
            print(f"Calendar features merged: {list(cal_feats.columns)}")
        except Exception as e:
            print(f"Calendar features skipped: {e}")

    return df


print("compute_features() ready")


In [ ]:
df_features = compute_features(df, pair="EURUSD")
FEATURE_COLS = [c for c in df_features.columns if c not in [
    "datetime", "open", "high", "low", "close", "volume",
    "sell_label", "buy_label", "y_agnostic", "split",
]]

print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"Date range: {df_features['datetime'].min()} -> {df_features['datetime'].max()}")
print(f"Rows: {len(df_features):,}")


## 5. Label Sweep — Tune `R_ENVELOPE`

Before committing to a model, sweep over R values to understand the label behavior:

- **Base rate** — should be near 50% for all R (symmetry check)
- **Timeout rate** — fraction of bars where neither target is hit within FORWARD_BARS
- **ROC-AUC ceiling** — what a perfect model would score at each R

The sweep helps pick an R value that produces enough labels (low timeout) while keeping
the problem hard enough that a model with real signal is distinguishable from chance.


In [ ]:
def compute_agnostic_labels_scan(data: pd.DataFrame, r: float, forward_bars: int):
    """
    Compute direction-agnostic labels for a given R value.

    Returns
    -------
    (labels, timeout_rate, base_rate)
      labels       : np.ndarray of 0/1  (1 = UP target hit first)
      timeout_rate : fraction where neither target hit
      base_rate    : fraction of resolved bars that went UP
    """
    close = data["close"].values
    high  = data["high"].values
    low   = data["low"].values
    atr   = data["atr_14"].values
    n     = len(data)

    labels = np.full(n, np.nan)

    for i in range(n - forward_bars):
        entry    = close[i]
        tp_up    = entry + atr[i] * r
        tp_down  = entry - atr[i] * r

        result = -1   # -1 = unresolved (timeout)
        for j in range(1, forward_bars + 1):
            k = i + j
            up_hit   = high[k] >= tp_up
            down_hit = low[k]  <= tp_down

            if up_hit and down_hit:
                result = -1   # both hit same bar — ambiguous, treat as timeout
                break
            elif up_hit:
                result = 1   # UP hit first
                break
            elif down_hit:
                result = 0   # DOWN hit first
                break

        labels[i] = result

    resolved = labels[labels >= 0]
    timeout_rate = (labels == -1).sum() / n
    base_rate = resolved.mean() if len(resolved) > 0 else 0.5

    return labels, timeout_rate, base_rate


# ── Sweep over R values ──────────────────────────────────────────────────────
R_VALUES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
print(f"{'R':>6}{'Timeout%':>10}{'BaseRate(UP%)':>14}{'Resolved':>10}")
print("-" * 42)
for r in R_VALUES:
    _, timeout, base = compute_agnostic_labels_scan(df_features, r, FORWARD_BARS)
    print(f"{r:>6.2f}{timeout*100:>10.1f}{base*100:>14.1f}{(1 - timeout)*100:>10.1f}")

# ── Visualization ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
results = []
for r in R_VALUES:
    _, to, ba = compute_agnostic_labels_scan(df_features, r, FORWARD_BARS)
    results.append((r, to, ba))
ax.plot([r[0] for r in results], [r[2]*100 for r in results], "o-", label="UP base rate", lw=2)
ax.plot([r[0] for r in results], [(1 - r[1])*100 for r in results], "s--", label="Resolution %", lw=2)
ax.axhline(50, color="gray", ls=":", label="50/50 coin flip")
ax.set_xlabel("R (ATR multiplier)")
ax.set_ylabel("%")
ax.set_title(f"Direction-Agnostic Label Sweep — EURUSD H1 (fw={FORWARD_BARS})")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nR_ENVELOPE (config) = {R_ENVELOPE}")
print("Run the next cell to generate labels at this R value.")


### 5.1 Generate Labels

Generate symmetrical UP/DOWN labels at the configured `R_ENVELOPE` value and attach
them to the feature DataFrame.


In [ ]:
def generate_agnostic_labels(
    data: pd.DataFrame,
    forward_bars: int,
    r_envelope: float,
) -> pd.DataFrame:
    """
    Generate direction-agnostic labels for each H1 bar.

    For each bar t:
        entry    = close[t]
        tp_up    = entry + ATR[t] * r_envelope     (upper target)
        tp_down  = entry - ATR[t] * r_envelope     (lower target)

        Scan forward `forward_bars` bars. If high[k] >= tp_up occurs before
        low[k] <= tp_down, label = 1 (UP). Otherwise label = 0 (DOWN or timeout).

    Parameters
    ----------
    data         : DataFrame with columns close, high, low, atr_14 (sorted)
    forward_bars : how many H1 bars ahead to scan
    r_envelope   : symmetrical risk distance in ATR units

    Returns
    -------
    DataFrame with new column 'y_agnostic' (1.0 or 0.0).
    The last `forward_bars` rows are dropped (incomplete forward window).
    """
    df = data.copy()
    n = len(df)

    close = df["close"].values
    high  = df["high"].values
    low   = df["low"].values
    atr   = df["atr_14"].values

    labels = np.full(n, np.nan)
    n_up, n_down, n_timeout = 0, 0, 0

    for i in range(n - forward_bars):
        entry    = close[i]
        tp_up    = entry + atr[i] * r_envelope
        tp_down  = entry - atr[i] * r_envelope

        result = -1
        for j in range(1, forward_bars + 1):
            k = i + j
            up_hit   = high[k] >= tp_up
            down_hit = low[k]  <= tp_down

            if up_hit and down_hit:
                result = -1
                break
            elif up_hit:
                result = 1
                break
            elif down_hit:
                result = 0
                break

        if result == 1:
            n_up += 1
            labels[i] = 1.0
        elif result == 0:
            n_down += 1
            labels[i] = 0.0
        else:
            n_timeout += 1
            labels[i] = 0.0   # conservative: timeout classified as DOWN

    df["y_agnostic"] = labels

    n_before = len(df)
    df.dropna(subset=["y_agnostic"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    total_resolved = n_up + n_down
    print(f"Labels: UP={n_up:,} ({n_up/total_resolved*100:.1f}%)  "
          f"DOWN={n_down:,} ({n_down/total_resolved*100:.1f}%)  "
          f"Timeout={n_timeout:,}  "
          f"Dropped tail={n_before - len(df)}")

    return df


df_features = generate_agnostic_labels(df_features, FORWARD_BARS, R_ENVELOPE)
print(f"\nFinal rows with labels   : {len(df_features):,}")
print(f"Base rate (UP=1)         : {df_features['y_agnostic'].mean()*100:.1f}%")
print(f"Majority baseline        : {max(df_features['y_agnostic'].mean(), 1 - df_features['y_agnostic'].mean())*100:.1f}%")
print("→ model must beat coin-flip on ROC-AUC (baseline = 0.50)")


## 6. Chronological Split — Train / Validation / Test

Time-based split with **no shuffling** (same discipline as the EURUSD notebooks).
The label was assigned *before* splitting — `y_agnostic` is already the outcome,
so there is no leakage in labelling before splitting.

| Set | Period | Purpose |
|---|---|---|
| **Train** | 2020–2025 H1 | model fitting |
| **Validation** | 2025 H2 | model selection + threshold calibration |
| **Test** | 2026+ | sealed hold-out |


In [ ]:
def assign_split(dt):
    if dt < pd.Timestamp(TRAIN_END):
        return "train"
    elif dt < pd.Timestamp(VAL_END):
        return "val"
    else:
        return "test"

df_features["split"] = df_features["datetime"].apply(assign_split)

df_train = df_features[df_features["split"] == "train"].reset_index(drop=True)
df_val   = df_features[df_features["split"] == "val"].reset_index(drop=True)
df_test  = df_features[df_features["split"] == "test"].reset_index(drop=True)

print("=== Class balance (y_agnostic: 1=UP, 0=DOWN) ===")
print(f"{'Split':<7}{'Total':>9}{'UP=1':>8}{'DOWN=0':>9}{'%UP':>8}")
print("-" * 43)
for name, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    n = len(d); up = int(d["y_agnostic"].sum())
    print(f"{name:<7}{n:>9,}{up:>8,}{n - up:>9,}{up / n * 100:>7.1f}%")


## 7. Feature Selection

**Method — noise features + voting system** (identical to the EURUSD notebooks).

We inject random *noise* features alongside the real features, train three models, and
read their importances via permutation importance (held-out, PR-AUC). Real features that
score **below the noise** are dropped. Everything is fit on **`df_train` only**.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

X_train = df_train[FEATURE_COLS].fillna(0)
y_train = df_train["y_agnostic"].astype(int)

print(f"Real features : {X_train.shape[1]}")
print(f"X_train shape : {X_train.shape}")
print(f"Positive rate : {y_train.mean()*100:.1f}%  (UP)")

np.random.seed(42)
n = len(X_train)
noise = {
    "noise_gaussian_1":  np.random.normal(0, 1, n),
    "noise_gaussian_2":  np.random.normal(0, 2, n),
    "noise_gaussian_3":  np.random.normal(5, 1, n),
    "noise_uniform_1":   np.random.uniform(-1, 1, n),
    "noise_uniform_2":   np.random.uniform(-10, 10, n),
    "noise_poisson_1":   np.random.poisson(3, n),
    "noise_poisson_2":   np.random.poisson(6, n),
    "noise_random_walk": np.cumsum(np.random.normal(0, 1, n)),
    "noise_sinusoidal":  np.sin(np.linspace(0, 10, n)) + np.random.normal(0, 0.1, n),
}
noise_features = list(noise.keys())
X_noise = X_train.copy()
for name, values in noise.items():
    X_noise[name] = values
print(f"Noise features added : {len(noise_features)}")
print(f"X_noise shape        : {X_noise.shape}")


In [ ]:
from sklearn.inspection import permutation_importance

n_fit = int(len(X_noise) * 0.75)
X_fit, X_perm = X_noise.iloc[:n_fit], X_noise.iloc[n_fit:]
y_fit, y_perm = y_train.iloc[:n_fit], y_train.iloc[n_fit:]

print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, max_features="sqrt", class_weight="balanced",
    random_state=42, n_jobs=3)
rf.fit(X_fit, y_fit)

print("Training LightGBM...")
lgbm = lgb.LGBMClassifier(n_estimators=250, max_depth=10, learning_rate=0.05,
    num_leaves=63, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, verbose=-1)
lgbm.fit(X_fit, y_fit)

print("Training Logistic Regression...")
scaler = StandardScaler()
X_fit_scaled = pd.DataFrame(scaler.fit_transform(X_fit.fillna(0)), columns=X_fit.columns)
logreg = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=42)
logreg.fit(X_fit_scaled, y_fit)

# Permutation importance (held-out, ROC-AUC) for trees; |coef| for linear
rf_imp = permutation_importance(rf, X_perm, y_perm, scoring="roc_auc",
    n_repeats=10, random_state=42, n_jobs=3).importances_mean
lgbm_imp = permutation_importance(lgbm, X_perm, y_perm, scoring="roc_auc",
    n_repeats=10, random_state=42, n_jobs=3).importances_mean
logreg_imp = np.abs(logreg.coef_).ravel()

def norm01(v):
    lo, hi = v.min(), v.max()
    return (v - lo) / (hi - lo) if hi > lo else np.zeros_like(v)

rf_imp, lgbm_imp, logreg_imp = norm01(rf_imp), norm01(lgbm_imp), norm01(logreg_imp)
print("All 3 models trained; importances normalised to [0,1]")


In [ ]:
imp_df = pd.DataFrame({
    "feature": X_noise.columns,
    "rf_imp": rf_imp, "lgbm_imp": lgbm_imp, "logreg_imp": logreg_imp,
})
imp_df["is_noise"] = imp_df["feature"].isin(noise_features)
imp_df["avg_imp"] = (imp_df["rf_imp"] + imp_df["lgbm_imp"] + imp_df["logreg_imp"]) / 3
imp_df = imp_df.sort_values("avg_imp", ascending=False).reset_index(drop=True)

VOTING_PERCENTILE = 45
rf_thr     = np.percentile(rf_imp, VOTING_PERCENTILE)
lgbm_thr   = np.percentile(lgbm_imp, VOTING_PERCENTILE)
logreg_thr = np.percentile(logreg_imp, VOTING_PERCENTILE)

imp_df["rf_vote"]     = (imp_df["rf_imp"]     >= rf_thr).astype(int)
imp_df["lgbm_vote"]   = (imp_df["lgbm_imp"]   >= lgbm_thr).astype(int)
imp_df["logreg_vote"] = (imp_df["logreg_imp"] >= logreg_thr).astype(int)
imp_df["total_votes"] = imp_df[["rf_vote", "lgbm_vote", "logreg_vote"]].sum(axis=1)

noise_df = imp_df[imp_df["is_noise"]]
real_df  = imp_df[~imp_df["is_noise"]]

noise_imp_mean  = noise_df["avg_imp"].mean()
noise_imp_std   = noise_df["avg_imp"].std()
noise_imp_p70   = np.percentile(noise_df["avg_imp"], 70)
noise_votes_max = noise_df["total_votes"].max()

best_noise_rank = noise_df.index.min()
strategies = {
    "better_than_best_noise": set(real_df[real_df.index < best_noise_rank]["feature"]),
    "above_noise_p70": set(real_df[real_df["avg_imp"] > noise_imp_p70]["feature"]),
    "more_votes_than_noise": set(real_df[real_df["total_votes"] > noise_votes_max]["feature"]),
    "statistical_threshold": set(real_df[real_df["avg_imp"] > noise_imp_mean + 0.5 * noise_imp_std]["feature"]),
    "vote_and_above_mean": set(real_df[(real_df["total_votes"] >= 1) & (real_df["avg_imp"] > noise_imp_mean)]["feature"]),
}

support = {}
for feats in strategies.values():
    for f in feats:
        support[f] = support.get(f, 0) + 1

MIN_STRATEGY_SUPPORT = 2
selected_features = sorted(
    [f for f, s in support.items() if s >= MIN_STRATEGY_SUPPORT and f not in noise_features],
    key=lambda f: imp_df.loc[imp_df["feature"] == f, "avg_imp"].iloc[0], reverse=True,
)

if len(selected_features) == 0:
    print("NO real feature beat noise — falling back to all real features.")
    selected_features = FEATURE_COLS

print(f"Selected features: {len(selected_features)} / {len(FEATURE_COLS)}")
print("\nTop 15 selected:")
print(imp_df[imp_df["feature"].isin(selected_features)][["feature", "avg_imp", "total_votes"]].head(15).to_string(index=False))

# Save
selected_report = imp_df[imp_df["feature"].isin(selected_features)]
selected_report.to_csv(FEATURES_DIR / "AGNOSTIC_H1_selected_features.csv", index=False)
imp_df.to_csv(FEATURES_DIR / "AGNOSTIC_H1_feature_importance_full.csv", index=False)
print(f"\nSaved to {FEATURES_DIR}/AGNOSTIC_H1_*")


## 8. Model Selection

- **Outer CV** — purged expanding TimeSeriesSplit (5-fold, 30-day embargo)
- **Inner CV** — TimeSeriesSplit (4-fold) for hyperparameter search
- **Recency weights** — recent years weighted more heavily
- **Optimization metric** — **ROC-AUC** (natural choice for this ~50/50 balanced problem)

Progression: LogReg (baseline) → RandomForest → XGBoost → LightGBM.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from xgboost import XGBClassifier

X_tr = df_train[selected_features].fillna(0)
y_tr = df_train["y_agnostic"].astype(int)
X_va = df_val[selected_features].fillna(0)
y_va = df_val["y_agnostic"].astype(int)
X_te = df_test[selected_features].fillna(0)
y_te = df_test["y_agnostic"].astype(int)

# ── Purged expanding TSS ────────────────────────────────────────────────────
train_dates = pd.to_datetime(df_train["datetime"])
n_train = len(df_train)
outer_cv, fold_boundaries = [], []
block_size = n_train // (OUTER_FOLDS + 1)

for fold in range(OUTER_FOLDS):
    tsr = block_size * (fold + 1)
    ter = block_size * (fold + 2) if fold < OUTER_FOLDS - 1 else n_train
    lt = train_dates.iloc[tsr - 1]
    pt = lt + pd.Timedelta(days=PURGE_DAYS)
    ts = tsr
    while ts < ter and train_dates.iloc[ts] < pt:
        ts += 1
    if ter - ts < 50:
        continue
    outer_cv.append((np.arange(0, ts), np.arange(ts, ter)))
    fold_boundaries.append((train_dates.iloc[ts], train_dates.iloc[ter - 1]))

# ── Recency weights ─────────────────────────────────────────────────────────
train_years = df_train["datetime"].dt.year.values
def time_decay_weights(years, decay=RECENCY_DECAY):
    ya = years.max() - years
    w = np.exp(-decay * ya)
    return w * len(w) / w.sum()
sample_weights = time_decay_weights(train_years)

pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

models = {
    "LogReg": {"model": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ]), "params": {}},
    "RandomForest": {"model": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=3), "params": {
        "n_estimators": [100, 200, 300], "max_depth": [10, 12, 15, 20],
        "min_samples_leaf": [5, 10, 20, 50], "max_features": ["sqrt", "log2", 0.5],
    }},
    "XGBoost": {"model": XGBClassifier(random_state=42, n_jobs=3, tree_method="hist",
        eval_metric="logloss", scale_pos_weight=pos_weight), "params": {
        "max_depth": [5, 7, 9, 11], "learning_rate": [0.05, 0.10, 0.20],
        "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9], "reg_lambda": [0.5, 1.0, 5.0],
    }},
    "LightGBM": {"model": lgb.LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=3, verbose=-1), "params": {
        "max_depth": [6, 8, 10, 12], "num_leaves": [15, 31, 63],
        "learning_rate": [0.05, 0.10, 0.20], "n_estimators": [100, 200, 300],
        "subsample": [0.8, 0.9, 1.0], "colsample_bytree": [0.7, 0.8, 0.9],
        "reg_lambda": [0.1, 0.5, 1.0, 5.0], "min_child_samples": [10, 20, 50],
    }},
}

INNER_CV = TimeSeriesSplit(n_splits=INNER_FOLDS)

print(f"Features : {len(selected_features)}")
print(f"Train    : {X_tr.shape}  (UP {y_tr.mean()*100:.1f}%)")
print(f"Val      : {X_va.shape}  (UP {y_va.mean()*100:.1f}%)")
print(f"Test     : {X_te.shape}  (sealed)")
print(f"Outer CV : {len(outer_cv)} folds, {PURGE_DAYS}-day embargo")
print(f"Inner CV : TimeSeriesSplit({INNER_FOLDS})")
print(f"Search   : {SEARCH_ITERS} iters | optimizing ROC-AUC")


In [ ]:
from sklearn.metrics import roc_auc_score

def nested_cv(model, params, X, y, weights, name):
    outer_scores, best_params_per_fold = [], []
    for fold, (tr, te) in enumerate(outer_cv, 1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        w_tr = weights[tr]
        if params:
            search = RandomizedSearchCV(model, params, n_iter=SEARCH_ITERS,
                cv=INNER_CV, scoring="roc_auc", n_jobs=3, random_state=42)
            search.fit(X_tr, y_tr, sample_weight=w_tr)
            best = search.best_estimator_
            best_params_per_fold.append(search.best_params_)
        else:
            best = model.fit(X_tr, y_tr)
            best_params_per_fold.append({})
        proba = best.predict_proba(X_te)[:, 1]
        outer_scores.append(roc_auc_score(y_te, proba))
        print(f"   fold {fold}/{len(outer_cv)}  ROC-AUC = {outer_scores[-1]:.3f}")
    print(f"   -> {name}: ROC-AUC {np.mean(outer_scores):.3f} +/- {np.std(outer_scores):.3f}")
    return {"name": name, "roc_auc_mean": np.mean(outer_scores),
            "roc_auc_std": np.std(outer_scores), "best_params_per_fold": best_params_per_fold}

cv_results = {}
for name, cfg in models.items():
    print(f"\n{name}")
    cv_results[name] = nested_cv(cfg["model"], cfg["params"], X_tr, y_tr, sample_weights, name)

comparison = pd.DataFrame([
    {"model": r["name"], "roc_auc": r["roc_auc_mean"], "std": r["roc_auc_std"]}
    for r in cv_results.values()
]).sort_values("roc_auc", ascending=False).reset_index(drop=True)

print("\n=== Model comparison (ROC-AUC) ===")
print(comparison.round(3).to_string(index=False))
print(f"\nCoin-flip baseline ROC-AUC = 0.500")
best_model_name = comparison.iloc[0]["model"]
print(f"Best model: {best_model_name}")


In [ ]:
from collections import Counter
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, accuracy_score)

USE_CALIBRATION = True
USE_ENSEMBLE    = True
CALIB_CV        = 3

def most_frequent_params(params_list):
    final = {}
    for key in {k for p in params_list for k in p}:
        vals = [p[key] for p in params_list if key in p]
        final[key] = Counter(vals).most_common(1)[0][0]
    return final

def build_tuned(name):
    est = models[name]["model"]
    params = most_frequent_params(cv_results[name]["best_params_per_fold"])
    if params: est.set_params(**params)
    return est, params

def make_calibrated(est):
    return CalibratedClassifierCV(est, method="isotonic", cv=CALIB_CV) if USE_CALIBRATION else est

if USE_ENSEMBLE:
    estimators = [(name, make_calibrated(build_tuned(name)[0])) for name in models]
    final_model = VotingClassifier(estimators=estimators, voting="soft", n_jobs=3)
    final_model.fit(X_tr, y_tr)
    best_model_name = "Ensemble"
    print("Final model = soft-vote ensemble of 4 calibrated models")
else:
    base, params = build_tuned(best_model_name)
    final_model = make_calibrated(base)
    try: final_model.fit(X_tr, y_tr, sample_weight=sample_weights)
    except TypeError: final_model.fit(X_tr, y_tr)
    print(f"Final model = {best_model_name}")

val_proba = final_model.predict_proba(X_va)[:, 1]   # P(up)
val_pred  = (val_proba >= 0.5).astype(int)

print(f"\n=== Validation (@0.5) — {best_model_name} ===")
print(f"ROC-AUC   : {roc_auc_score(y_va, val_proba):.3f}   (coin flip = 0.5)")
print(f"Accuracy  : {accuracy_score(y_va, val_pred):.3f}")
print(f"Precision : {precision_score(y_va, val_pred, zero_division=0):.3f}")
print(f"Recall    : {recall_score(y_va, val_pred, zero_division=0):.3f}")
print(f"F1        : {f1_score(y_va, val_pred, zero_division=0):.3f}")
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_va, val_pred))


### 8.1 Direction Decision Boundary

This is a **balanced** binary problem (~50/50), so the natural decision boundary is
**0.5** — predict UP when P(up) > 0.5, DOWN otherwise. There is no asymmetric threshold
to tune (that logic existed only for the imbalanced SELL problem).

The meaningful confidence metric is the **margin** `|P(up) − 0.5|` — how far the model
is from a coin flip. The sealed test reports directional accuracy at various margin
thresholds.


In [ ]:
from sklearn.calibration import calibration_curve

# Calibration on validation
frac_pos, mean_pred = calibration_curve(y_va, val_proba, n_bins=10)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(mean_pred, frac_pos, "s-", color="steelblue", label="agnostic model")
ax.plot([0, 1], [0, 1], "k:", label="perfectly calibrated")
ax.set_xlabel("Mean predicted P(up)"); ax.set_ylabel("Fraction of UP")
ax.set_title(f"EURUSD Agnostic — calibration (validation)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Sealed test evaluation (opened once) ──────────────────────────────────────
test_proba = final_model.predict_proba(X_te)[:, 1]   # P(up)
test_pred  = (test_proba >= 0.5).astype(int)

print("=" * 60)
print(f"SEALED TEST — {best_model_name}  (direction cutoff = 0.5)")
print("=" * 60)
print(f"ROC-AUC   : {roc_auc_score(y_te, test_proba):.3f}   (gate bar >= 0.55; coin flip = 0.5)")
print(f"Accuracy  : {accuracy_score(y_te, test_pred):.3f}")
print(f"Precision : {precision_score(y_te, test_pred, zero_division=0):.3f}")
print(f"Recall    : {recall_score(y_te, test_pred, zero_division=0):.3f}")
print(f"F1        : {f1_score(y_te, test_pred, zero_division=0):.3f}")
print(f"Base rate : {y_te.mean():.3f}   (P(up))")
print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_te, test_pred))

# ── Direction-margin sweep (the real confidence metric) ────────────────────
test_eval = df_test[["datetime"]].copy()
test_eval["y"]      = y_te.values
test_eval["proba"]  = test_proba
test_eval["pred"]   = test_pred
test_eval["margin"] = np.abs(test_proba - 0.5)

print("\n=== Direction accuracy vs confidence margin (sealed test) ===")
print(f"{'margin >=':>10}{'n':>7}{'accuracy':>11}{'coverage':>10}")
print("-" * 40)
for m in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    keep = test_eval["margin"] >= m
    n_keep = int(keep.sum())
    if n_keep == 0:
        print(f"{m:>10.2f}{n_keep:>7}{'--':>11}{0.0:>10.1f}%")
        continue
    acc = accuracy_score(test_eval.loc[keep, "y"], test_eval.loc[keep, "pred"])
    cov = n_keep / len(test_eval)
    print(f"{m:>10.2f}{n_keep:>7}{acc:>11.3f}{cov*100:>9.1f}%")

# ── Calibration on test ─────────────────────────────────────────────────────
frac_pos2, mean_pred2 = calibration_curve(y_te, test_proba, n_bins=10)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(mean_pred2, frac_pos2, "s-", color="steelblue", label="agnostic model")
ax.plot([0, 1], [0, 1], "k:", label="perfectly calibrated")
ax.set_xlabel("Mean predicted P(up)"); ax.set_ylabel("Fraction of UP")
ax.set_title(f"EURUSD Agnostic — calibration (sealed test)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import joblib

GATE_THRESHOLD = 0.5  # balanced cutoff (P(up) > 0.5 = BUY, otherwise SELL)

model_path = MODELS_DIR / "EURUSD_H1_agnostic_Ensemble.joblib"
joblib.dump({
    "model": final_model,
    "features": selected_features,
    "threshold": GATE_THRESHOLD,
    "r_envelope": R_ENVELOPE,
    "forward_bars": FORWARD_BARS,
    "train_end": TRAIN_END,
    "val_end": VAL_END,
}, model_path)

print(f"Saved model bundle -> {model_path}")
print(f"  model        : {best_model_name}")
print(f"  features     : {len(selected_features)}")
print(f"  threshold    : {GATE_THRESHOLD}  (balanced cutoff)")
print(f"  R envelope   : {R_ENVELOPE}")


## Next Steps

1. **Run this notebook top-to-bottom** — the sealed test ROC-AUC answers the threshold
   question: does directional signal exist at the 6-hour horizon?
2. **If ROC-AUC >= 0.55** → the model is shadow-worthy. Integrate into the pipeline
   as `EURUSD_AGNOSTIC` (shadow only, direction-aware, 2-week monitoring).
3. **If ROC-AUC < 0.55** → directional signal is not detectable with our H1 technical
   feature set. The binding constraint is fundamental features (real yields, VIX), not
   model architecture.
4. **If shadow accuracy >= 55%** after 2 weeks → flip `shadow: false` and run live
   alongside the existing SELL pipeline.

### What the combiner notebook will do

A new `signal_combiner_agnostic.ipynb` will:
- Load the agnostic model bundle
- Evaluate directional accuracy over time (monthly walk-forward)
- Report UP-vs-DOWN precision by session, by margin
- Generate a Wilson CI for the directional accuracy
